# BirdCLEF 2026 — Ensemble Submission (v1 + v2)

Averages sigmoid outputs from two models trained on different mel spectrogram representations:
- **v1**: `multilabel_234/5` — ConvNeXt-Small, hop=512, librosa defaults
- **v2**: `multilabel_234_v2/1` — ECA-NFNet-L0, hop=320, n_mels=128, fmin=50, fmax=14000, n_fft=1024

In [ ]:
import json
import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torchvision import transforms as T
from fastai.vision.all import load_learner

warnings.filterwarnings('ignore', category=UserWarning, module='fastai')

In [ ]:
import kagglehub
path = kagglehub.competition_download('birdclef-2026')
print('Path to competition files:', path)

In [ ]:
# v1 model — ConvNeXt-Small, trained on default mel params (hop=512, librosa defaults)
V1_MODEL_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234/5/model_multilabel_234.pkl'
V1_VOCAB_PATH  = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234/5/vocab.json'
V1_HOP = 512

# v2 model — ECA-NFNet-L0, trained on new mel params
V2_MODEL_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234_v2/1/model_multilabel_234.pkl'
V2_VOCAB_PATH  = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234_v2/1/vocab.json'
V2_HOP    = 320
V2_N_MELS = 128
V2_FMIN   = 50
V2_FMAX   = 14000
V2_N_FFT  = 1024

TEST_DIR       = Path(path) / 'test_soundscapes'
TARGET_SIZE    = (224, 224)
CLIP_DURATION  = 5
SAMPLE_RATE    = 32000
BATCH_SIZE     = 64
STRIDE_DURATION = 2.5

In [ ]:
sample_sub = pd.read_csv(Path(path) / 'sample_submission.csv')
all_species = [c for c in sample_sub.columns if c != 'row_id']
assert len(all_species) == 234, f'expected 234, got {len(all_species)}'

learn_v1 = load_learner(V1_MODEL_PATH, cpu=True)
learn_v2 = load_learner(V2_MODEL_PATH, cpu=True)

for label, learn, vocab_path in [('v1', learn_v1, V1_VOCAB_PATH), ('v2', learn_v2, V2_VOCAB_PATH)]:
    vocab = list(learn.dls.vocab)
    assert vocab == all_species, f'{label} vocab does not match sample_submission column order'
    if Path(vocab_path).exists():
        assert json.load(open(vocab_path)) == vocab, f'{label} vocab.json disagrees with learner'
    print(f'{label} vocab ok ({len(vocab)} classes)')

learn_v1.model.eval()
learn_v2.model.eval()
device = next(learn_v1.model.parameters()).device
print(f'Running on: {device}')

In [ ]:
clip_length    = CLIP_DURATION * SAMPLE_RATE
stride_samples = int(STRIDE_DURATION * SAMPLE_RATE)

tfm = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def window_to_img(window):
    lo, hi = float(window.min()), float(window.max())
    arr = np.zeros_like(window, dtype=np.uint8) if hi == lo else (
        (window - lo) / (hi - lo) * 255
    ).astype(np.uint8)
    return Image.fromarray(arr).resize(TARGET_SIZE).convert('RGB')


def mel_to_submission_preds(learn, S_db, hop_length, n_samples):
    """Run model over all overlapping clips; aggregate into per-5s-window max predictions."""
    frames_per_clip = int(CLIP_DURATION * SAMPLE_RATE / hop_length)
    stride_frames   = int(STRIDE_DURATION * SAMPLE_RATE / hop_length)

    imgs = []
    k = 0
    while k * stride_samples + clip_length <= n_samples:
        window = S_db[:, k * stride_frames : k * stride_frames + frames_per_clip]
        imgs.append(window_to_img(window))
        k += 1
    n_overlapping = len(imgs)

    overlap_preds = []
    for i in range(0, n_overlapping, BATCH_SIZE):
        batch = torch.stack([tfm(img) for img in imgs[i:i + BATCH_SIZE]]).to(device)
        with torch.no_grad():
            logits = learn.model(batch)
        overlap_preds.append(torch.sigmoid(logits).cpu().numpy())
    overlap_preds = np.vstack(overlap_preds)  # (n_overlapping, 234)

    n_submission = n_samples // clip_length
    window_preds = []
    for j in range(n_submission):
        k_lo = max(0, 2 * j - 1)
        k_hi = min(n_overlapping - 1, 2 * j + 1)
        window_preds.append(overlap_preds[k_lo:k_hi + 1].max(axis=0))
    return np.array(window_preds)  # (n_submission, 234)

In [ ]:
test_files = sorted(TEST_DIR.glob('*.ogg'))
if not test_files:
    fallback_dir = Path(path) / 'train_soundscapes'
    test_files = sorted(fallback_dir.glob('*.ogg'))[:3]
    print(f'[dry-run] test_soundscapes empty, using {len(test_files)} train_soundscapes files')
else:
    print(f'Found {len(test_files)} test soundscape files')

row_ids   = []
all_preds = []

for soundscape in test_files:
    samples, _ = librosa.load(soundscape, sr=SAMPLE_RATE)
    n_samples   = len(samples)
    n_submission = n_samples // clip_length

    # v1 mel: default params (hop=512, librosa defaults for fmin/fmax/n_fft)
    S_db_v1 = librosa.power_to_db(
        librosa.feature.melspectrogram(y=samples, sr=SAMPLE_RATE, hop_length=V1_HOP),
        ref=np.max,
    )
    preds_v1 = mel_to_submission_preds(learn_v1, S_db_v1, V1_HOP, n_samples)

    # v2 mel: new params (hop=320, n_mels=128, fmin=50, fmax=14000, n_fft=1024)
    S_db_v2 = librosa.power_to_db(
        librosa.feature.melspectrogram(
            y=samples, sr=SAMPLE_RATE, hop_length=V2_HOP,
            n_mels=V2_N_MELS, fmin=V2_FMIN, fmax=V2_FMAX, n_fft=V2_N_FFT,
        ),
        ref=np.max,
    )
    preds_v2 = mel_to_submission_preds(learn_v2, S_db_v2, V2_HOP, n_samples)

    preds_ensemble = (preds_v1 + preds_v2) / 2.0
    all_preds.append(preds_ensemble)
    for j in range(n_submission):
        row_ids.append(f'{soundscape.stem}_{(j + 1) * CLIP_DURATION}')

preds_np = np.vstack(all_preds)
assert preds_np.shape == (len(row_ids), 234), preds_np.shape
print(f'Inference complete: {preds_np.shape[0]} clips across {len(test_files)} files')

In [ ]:
submission = pd.DataFrame(preds_np, columns=all_species)
submission.insert(0, 'row_id', row_ids)
assert list(submission.columns) == ['row_id'] + all_species
submission.to_csv('submission.csv', index=False)
print(f'Done. {len(submission)} rows written.')
submission.head()